# Nextbus Feeder (Fabric Notebook)\n\nRuns a single Nextbus polling cycle and emits CloudEvents into the bound Fabric Event Stream custom endpoint.

In [ ]:
# === PARAMETERS (overwritten by deploy-feeder-notebook.ps1) ===\nEVENTSTREAM_NAME = "nextbus-ingest"\nSTATE_FILE       = "/lakehouse/default/Files/feeder-state/nextbus/state.json"\nWORKSPACE_ID     = ""\nAGENCY           = "sf-muni"\nROUTE            = "*"

## Look up Event Stream connection string

In [ ]:
import os, time, json\nimport requests\n\nFABRIC_API = 'https://api.fabric.microsoft.com/v1'\n\ndef _get_pbi_token():\n    try:\n        import notebookutils\n        return notebookutils.credentials.getToken('pbi')\n    except Exception:\n        from notebookutils import mssparkutils\n        return mssparkutils.credentials.getToken('pbi')\n\ndef _resolve_workspace_id(explicit: str) -> str:\n    if explicit:\n        return explicit\n    try:\n        import notebookutils\n        ctx = notebookutils.runtime.context\n        return ctx['currentWorkspaceId']\n    except Exception:\n        pass\n    from notebookutils import mssparkutils\n    ctx = json.loads(mssparkutils.runtime.context)\n    return ctx['currentWorkspaceId']\n\ndef lookup_es_connection_string(workspace_id: str, eventstream_name: str) -> str:\n    token = _get_pbi_token()\n    headers = {'Authorization': 'Bearer ' + token}\n    es_list = requests.get(f'{FABRIC_API}/workspaces/{workspace_id}/eventstreams', headers=headers, timeout=30)\n    es_list.raise_for_status()\n    es = next((x for x in es_list.json().get('value', []) if x['displayName'] == eventstream_name), None)\n    if not es:\n        raise RuntimeError(f"Event Stream '{eventstream_name}' not found in workspace {workspace_id}.")\n    topo = requests.get(f'{FABRIC_API}/workspaces/{workspace_id}/eventstreams/{es["id"]}/topology', headers=headers, timeout=30)\n    topo.raise_for_status()\n    src = next((s for s in topo.json().get('sources', []) if s.get('type') == 'CustomEndpoint'), None)\n    if not src:\n        raise RuntimeError('Event Stream has no CustomEndpoint source.')\n    for attempt in range(5):\n        conn = requests.get(f'{FABRIC_API}/workspaces/{workspace_id}/eventstreams/{es["id"]}/sources/{src["id"]}/connection', headers=headers, timeout=30)\n        conn.raise_for_status()\n        cs = conn.json().get('accessKeys', {}).get('primaryConnectionString')\n        if cs:\n            return cs\n        time.sleep(min(15, 3 * (attempt + 1)))\n    raise RuntimeError('Could not retrieve Event Stream primary connection string.')\n\nws_id = _resolve_workspace_id(WORKSPACE_ID)\ncs = lookup_es_connection_string(ws_id, EVENTSTREAM_NAME)\nprint(f'Workspace: {ws_id}')\nprint(f'Event Stream: {EVENTSTREAM_NAME}')\nprint(f'Connection ready: length={len(cs)}')

## Run one polling cycle

In [ ]:
import datetime, pathlib, sys, time, traceback, threading\nfrom urllib.parse import parse_qsl\n\nLOG_PATH = '/lakehouse/default/Files/feeder-state/nextbus/last-run.log'\npathlib.Path(LOG_PATH).parent.mkdir(parents=True, exist_ok=True)\n\ndef _log(msg):\n    line = f'[{datetime.datetime.utcnow().isoformat()}Z] {msg}'\n    print(line)\n    with open(LOG_PATH, 'a', encoding='utf-8') as f:\n        f.write(line + '\n')\n\ndef _event_hub_name_from_connection_string(connection_string: str) -> str:\n    kv = dict(parse_qsl(connection_string.replace(';', '&')))\n    return kv.get('EntityPath', '')\n\nwith open(LOG_PATH, 'w', encoding='utf-8') as f:\n    f.write('')\n\ntry:\n    pathlib.Path(STATE_FILE).parent.mkdir(parents=True, exist_ok=True)\n    os.environ['FEED_CONNECTION_STRING'] = cs\n    os.environ['AGENCY'] = AGENCY\n    os.environ['ROUTE'] = ROUTE\n    event_hub_name = _event_hub_name_from_connection_string(cs)\n    if not event_hub_name:\n        raise RuntimeError('Event Stream connection string does not include EntityPath.')\n    os.environ['FEED_EVENT_HUB_NAME'] = event_hub_name\n    _log(f'Using feed Event Hub: {event_hub_name}')\n\n    from nextbus import nextbus as feeder\n    _log(f'Imported feeder from: {feeder.__file__}')\n\n    _err = []\n    def _worker():\n        feed_client = feeder.EventHubProducerClient.from_connection_string(cs, eventhub_name=event_hub_name)\n        try:\n            feeder.poll_and_submit_route_config(feed_client, AGENCY)\n            feeder.poll_and_submit_schedule(feed_client, AGENCY)\n            feeder.poll_and_submit_messages(feed_client, AGENCY)\n            feeder.poll_and_submit_vehicle_locations(feed_client, AGENCY, ROUTE, time.time())\n        except BaseException as exc:\n            _err.append(exc)\n        finally:\n            feed_client.close()\n\n    t = threading.Thread(target=_worker, daemon=True)\n    t.start()\n    t.join()\n    if _err:\n        raise _err[0]\n\n    _log('Cycle complete.')\n    try:\n        import notebookutils\n        notebookutils.notebook.exit('OK')\n    except Exception:\n        pass\nexcept Exception as exc:\n    tb = traceback.format_exc()\n    _log(f'FAILED: {exc}\n{tb}')\n    try:\n        import notebookutils\n        notebookutils.notebook.exit(f'FAIL: {exc}')\n    except Exception:\n        pass\n    raise